# CUM Series 16d-f — TD Replication + Structural Slicing + 124M Scale

Three phases:
- **16d**: Replicate td mode (3 runs of QKV td d=-1.0)
- **16e**: Plain per-head + out_proj WITHOUT td (isolate structural contribution)
- **16f**: Scale test at 124M params (GPT-2 Small / OpenWebText)

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# --- Data: TinyShakespeare (1.2M) + OpenWebText (124M) ---
DATA_DIR = 'benchmarks/tier0/data'

# TinyShakespeare for 1.2M model
TINY_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(TINY_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        TINY_PATH)
with open(TINY_PATH, 'r') as f:
    TEXT = f.read()
print(f'TinyShakespeare: {len(TEXT):,} chars')

# OpenWebText for 124M model (~9B tokens, we cache 500M for speed)
import tiktoken
OWT_PATH = os.path.join(DATA_DIR, 'openwebtext_tokens_500m.pt')
TOKEN_BUDGET = 500_000_000  # 500M tokens — enough for 50k+ steps

if os.path.exists(OWT_PATH):
    print('Loading cached OpenWebText tokens...')
    all_tokens = torch.load(OWT_PATH)
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Downloading OpenWebText (streaming, first 500M tokens)...')
    from datasets import load_dataset
    ds = load_dataset('openwebtext', split='train', streaming=True)
    enc = tiktoken.get_encoding('gpt2')
    
    token_chunks = []
    total = 0
    for i, example in enumerate(ds):
        tokens = enc.encode(example['text'])
        token_chunks.extend(tokens)
        total += len(tokens)
        if i % 50000 == 0:
            print(f'  {total/1e6:.1f}M tokens from {i} docs...')
        if total >= TOKEN_BUDGET:
            break
    
    all_tokens = torch.tensor(token_chunks[:TOKEN_BUDGET], dtype=torch.long)
    torch.save(all_tokens, OWT_PATH)
    print(f'Cached {len(all_tokens)/1e6:.0f}M tokens to {OWT_PATH}')

print(f'OpenWebText tokens: {len(all_tokens)/1e6:.1f}M')
VOCAB_SIZE_BPE = 50257

In [ ]:
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum.tensor import PerHeadMuon, PerHeadBlendMuon

# 1.2M config
MODEL_CFG_SMALL = dict(d_model=128, n_heads=4, n_layers=4, d_ff=512, ctx_len=256)
# 124M config (GPT-2 Small)
MODEL_CFG_124M = dict(d_model=768, n_heads=12, n_layers=12, d_ff=3072, ctx_len=256)

BATCH_SIZE = 32
TOTAL_STEPS = 2000
WARMUP_STEPS = 200
EVAL_EVERY = 250
BASE_LR = 0.02
SEED = 42

test_small = MicroGPT(vocab_size=65, **MODEL_CFG_SMALL)
print(f'Small model: {sum(p.numel() for p in test_small.parameters())/1e6:.1f}M params')
del test_small

test_large = MicroGPT(vocab_size=VOCAB_SIZE_BPE, **MODEL_CFG_124M)
print(f'124M model: {sum(p.numel() for p in test_large.parameters())/1e6:.1f}M params')
del test_large
print('Imports OK')

In [ ]:
class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y


class TokenDataset:
    def __init__(self, tokens, ctx_len, device, val_frac=0.01):
        self.ctx_len = ctx_len
        self.vocab_size = VOCAB_SIZE_BPE
        n = int(len(tokens) * (1 - val_frac))
        self.train_data = tokens[:n].to(device)
        self.val_data = tokens[n:].to(device)

    def get_batch(self, batch_size, split='train'):
        data = self.train_data if split == 'train' else self.val_data
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y


dataset_small = CharDataset(TEXT, ctx_len=MODEL_CFG_SMALL['ctx_len'], device=DEVICE)
dataset_124m = TokenDataset(all_tokens, ctx_len=MODEL_CFG_124M['ctx_len'], device=DEVICE)
print(f'Small dataset on {DEVICE}: vocab={dataset_small.vocab_size}')
print(f'124M dataset on {DEVICE}: vocab={dataset_124m.vocab_size}, {len(dataset_124m.train_data)/1e6:.1f}M train tokens')

In [ ]:
class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def train_one(name, optimizers, model, dataset,
              total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE,
              warmup_steps=WARMUP_STEPS, base_lr=BASE_LR, eval_every=EVAL_EVERY):
    model.train()
    trajectory = []

    # Warmup torch.compile
    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        for opt in optimizers:
            opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for step in range(total_steps):
        current_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        for opt in optimizers:
            for g in opt.param_groups:
                if isinstance(opt, torch.optim.AdamW):
                    g['lr'] = get_lr(step, total_steps, warmup_steps, 3e-4)
                else:
                    g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        for opt in optimizers:
            opt.zero_grad()
        loss.backward()
        for opt in optimizers:
            opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
def split_params(model, n_slices=1, col_slices=1, all_sliced=False):
    """Split model params into groups for PerHeadBlendMuon."""
    raw = model._orig_mod if hasattr(model, '_orig_mod') else model
    qkv_params = []
    outproj_params = []
    mlp_up_params = []
    mlp_down_params = []
    other_hidden = []
    other_params = []

    for block in raw.blocks:
        for name, p in block.named_parameters():
            if p.ndim == 2:
                if 'qkv' in name:
                    qkv_params.append(p)
                elif 'out_proj' in name:
                    outproj_params.append(p)
                elif 'up' in name:
                    mlp_up_params.append(p)
                elif 'down' in name:
                    mlp_down_params.append(p)
                else:
                    other_hidden.append(p)
            else:
                other_params.append(p)

    hidden_set = set(id(p) for block in raw.blocks for p in block.parameters())
    for p in raw.parameters():
        if id(p) not in hidden_set:
            other_params.append(p)

    groups = []
    if qkv_params:
        groups.append({'params': qkv_params, 'n_slices': n_slices, 'col_slices': 1})
    if outproj_params:
        groups.append({'params': outproj_params, 'n_slices': 1, 'col_slices': col_slices})
    if all_sliced and n_slices > 1:
        if mlp_up_params:
            groups.append({'params': mlp_up_params, 'n_slices': n_slices, 'col_slices': 1})
        if mlp_down_params:
            groups.append({'params': mlp_down_params, 'n_slices': 1, 'col_slices': n_slices})
    else:
        remaining = mlp_up_params + mlp_down_params
        if remaining:
            groups.append({'params': remaining, 'n_slices': 1, 'col_slices': 1})
    if other_hidden:
        groups.append({'params': other_hidden, 'n_slices': 1, 'col_slices': 1})

    return groups, other_params


def make_model_and_opts(dataset, cfg, model_cfg=None, scale='small'):
    if model_cfg is None:
        model_cfg = MODEL_CFG_SMALL
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **model_cfg).to(DEVICE)
    model = torch.compile(model, mode='reduce-overhead')
    raw = model._orig_mod if hasattr(model, '_orig_mod') else model

    n_heads = model_cfg['n_heads']
    t = cfg['type']

    if t == 'Muon+AdamW':
        hidden_2d = raw.get_hidden_2d_params()
        other = raw.get_other_params()
        muon = SimpleMuon(hidden_2d, lr=BASE_LR, ns_steps=5)
        adam = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)
        optimizers = [muon, adam]

    elif t == 'PerHeadBlend':
        n_slices = cfg.get('n_slices', n_heads)
        col_slices = cfg.get('col_slices', 1)
        mode = cfg.get('mode', 'plain')
        deriv = cfg.get('deriv', -1.0)
        all_sliced = cfg.get('all_sliced', False)

        groups, other_params = split_params(model, n_slices, col_slices, all_sliced)
        ph = PerHeadBlendMuon(
            groups, lr=BASE_LR, ns_steps=5,
            mode=mode, deriv=deriv,
            td_lambda=0.5, input_blend_beta=0.5, input_blend_alpha=0.15,
        )
        adam = torch.optim.AdamW(other_params, lr=3e-4, weight_decay=0.01)
        optimizers = [ph, adam]

    else:
        raise ValueError(f'Unknown: {t}')

    return model, optimizers


def run_all(configs, dataset, model_cfg=None, scale='small'):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, optimizers = make_model_and_opts(dataset, cfg, model_cfg, scale)
            val_loss, traj, elapsed = train_one(name, optimizers, model, dataset)
            results.append(dict(name=name, type=cfg['type'], val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, type=cfg['type'], val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


def show_results(results, series='16d'):
    muon = next((r['val_loss'] for r in results if r['type'] == 'Muon+AdamW'), None)
    print(f'\n## Series {series} Results\n')
    print(f'| Config | Val Loss | vs Muon+AdamW | Time |')
    print(f'|--------|----------|---------------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r.get('error'):
            print(f'| {r["name"]} | FAILED | -- | {r["error"][:30]} |')
            continue
        vm = f'{r["val_loss"] - muon:+.4f}' if muon else '--'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vm} | {r["elapsed"]:.0f}s |')


print('Factory ready')

---
## 16d: TD Mode Replication (3 runs)

16b showed td d=-1.0 at -0.032 vs Muon. Is that as lucky as 15b's -0.025 was?
3 runs of QKV-only td to nail down the true delta.

In [ ]:
CONFIGS_16D = [
    {'name': 'Muon+AdamW', 'type': 'Muon+AdamW'},
    {'name': 'PerHead QKV td run1', 'type': 'PerHeadBlend', 'n_slices': 4, 'mode': 'td', 'deriv': -1.0},
    {'name': 'PerHead QKV td run2', 'type': 'PerHeadBlend', 'n_slices': 4, 'mode': 'td', 'deriv': -1.0},
    {'name': 'PerHead QKV td run3', 'type': 'PerHeadBlend', 'n_slices': 4, 'mode': 'td', 'deriv': -1.0},
]

results_16d = run_all(CONFIGS_16D, dataset_small, MODEL_CFG_SMALL)
show_results(results_16d, '16d: TD Replication')

---
## 16e: Plain Per-Head + Out_Proj (no blending)

Isolate the structural slicing contribution WITHOUT td/blending.
How much does out_proj col-slicing add on top of plain per-head QKV?

In [ ]:
CONFIGS_16E = [
    {'name': 'Muon+AdamW', 'type': 'Muon+AdamW'},
    {'name': 'PerHead QKV plain', 'type': 'PerHeadBlend', 'n_slices': 4, 'mode': 'plain'},
    {'name': 'PerHead QKV+outproj plain', 'type': 'PerHeadBlend', 'n_slices': 4, 'col_slices': 4, 'mode': 'plain'},
    {'name': 'PerHead QKV+outproj td', 'type': 'PerHeadBlend', 'n_slices': 4, 'col_slices': 4, 'mode': 'td', 'deriv': -1.0},
]

results_16e = run_all(CONFIGS_16E, dataset_small, MODEL_CFG_SMALL)
show_results(results_16e, '16e: Structural Slicing Isolation')

---
## 16f: 124M Scale Test (GPT-2 Small / OpenWebText)

12 slices (per-head) gave 0.25:1 ratio = WORSE than original. Fewer slices = more square:
- **3 slices** (Q/K/V): (768, 768) = 1:1 perfectly square
- **4 slices**: (576, 768) = 0.75:1 (matches 1.2M sweet spot)

Per-head ratio = 3/n_slices. Need n_slices ≤ ~4 for good aspect ratio.

In [ ]:
CONFIGS_16F = [
    {'name': 'Muon+AdamW', 'type': 'Muon+AdamW'},
    {'name': 'PerHead 3s td (Q/K/V)', 'type': 'PerHeadBlend', 'n_slices': 3, 'mode': 'td', 'deriv': -1.0},
    {'name': 'PerHead 4s td', 'type': 'PerHeadBlend', 'n_slices': 4, 'mode': 'td', 'deriv': -1.0},
]

results_16f = run_all(CONFIGS_16F, dataset_124m, MODEL_CFG_124M, scale='124m')
show_results(results_16f, '16f: 124M Scale Test')